In [1]:
import sys
sys.path.append(r'C:\traffic-demand-final\pipelining\src')
import os
import pandas as pd
import numpy as np
from config import *

train_df = pd.read_csv(os.path.join(PROC_DIR, 'train_step05.csv'))
test_df = pd.read_csv(os.path.join(PROC_DIR, 'test_step05.csv'))

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

print("\nColumns in train:")
print(train_df.columns.tolist())


Train shape: (77299, 20)
Test shape: (41778, 19)

Columns in train:
['day', 'demand', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'hour', 'minute', 'road_type_ord', 'weather_severity', 'latitude', 'longitude', 'geohash_target_enc', 'geohash_prefix_enc', 'hour_sin', 'hour_cos', 'is_peak_hour', 'is_business_hour', 'is_night', 'hour_squared']


In [2]:
train_df['road_capacity'] = train_df['road_type_ord'] * train_df['NumberofLanes']
test_df['road_capacity'] = test_df['road_type_ord'] * test_df['NumberofLanes']

print(f"road_capacity: min={train_df['road_capacity'].min()}, max={train_df['road_capacity'].max()}, mean={train_df['road_capacity'].mean():.2f}")

print("\nMean road_capacity per combination:")
print(pd.crosstab(train_df['road_type_ord'], train_df['NumberofLanes'], values=train_df['road_capacity'], aggfunc='mean'))

print("\nMean demand per road_capacity (sorted):")
print(train_df.groupby('road_capacity')['demand'].mean().sort_index())


road_capacity: min=0.0, max=10.0, mean=0.38

Mean road_capacity per combination:
NumberofLanes    1    2    3    4     5
road_type_ord                          
0.0            0.0  0.0  0.0  0.0   0.0
1.0            1.0  NaN  NaN  NaN   NaN
2.0            NaN  4.0  6.0  8.0  10.0

Mean demand per road_capacity (sorted):
road_capacity
0.0     0.057562
1.0     0.273164
4.0     0.619695
6.0     0.613320
8.0     0.603511
10.0    0.607156
Name: demand, dtype: float64


### Conclusion
The `road_capacity` feature represents the physical throughput limit of a road segment. It comes from the bivariate analysis which showed that both higher road types and more lanes individually correlated with higher demand. Multiplying `road_type_ord` (the hierarchical classification of the road) by `NumberofLanes` (the physical width) captures the combined theoretical capacity better than either feature alone, creating a single monotonic proxy for how much traffic a road can handle.

In [3]:
train_df['road_combo'] = train_df['LargeVehicles'].astype(str) + '_' + train_df['Landmarks'].astype(str)
test_df['road_combo'] = test_df['LargeVehicles'].astype(str) + '_' + test_df['Landmarks'].astype(str)

print("Value counts of road_combo in train:")
print(train_df['road_combo'].value_counts())

print("\nMean demand per road_combo (sorted by demand descending):")
print(train_df.groupby('road_combo')['demand'].mean().sort_values(ascending=False))

print(f"\nNulls in train road_combo: {train_df['road_combo'].isnull().sum()}")


Value counts of road_combo in train:
road_combo
0_1    27197
1_1    24845
0_0    23476
1_0     1781
Name: count, dtype: int64

Mean demand per road_combo (sorted by demand descending):
road_combo
1_0    0.613137
1_1    0.097427
0_1    0.088574
0_0    0.057085
Name: demand, dtype: float64

Nulls in train road_combo: 0


### Conclusion
The `road_combo` categories represent four physical environments:
*   `1_1`: Large vehicles allowed, landmarks present (high commercial/industrial activity area).
*   `1_0`: Large vehicles allowed, no landmarks (highway or standard arterial road).
*   `0_1`: Large vehicles not allowed, landmarks present (dense urban, historical, or restricted commercial areas).
*   `0_0`: Large vehicles not allowed, no landmarks (quiet residential or local roads).

The combination `1_1` typically exhibits the highest demand because it represents major economic hubs where both freight/commercial traffic (large vehicles) and significant passenger destinations (landmarks) overlap. The multivariate analysis revealed that the joint effect of these two variables creates a clear hierarchical ordering of demand that is much stronger than looking at either flag in isolation.

In [4]:
cap_summary = train_df[['road_capacity']].agg(['nunique', 'min', 'max', lambda x: x.isnull().sum()]).T
cap_summary.rename(columns={'<lambda>': 'null_count'}, inplace=True)
print("Numeric Feature Summary:")
display(cap_summary)

combo_summary = train_df[['road_combo']].agg(['nunique', lambda x: x.isnull().sum()]).T
combo_summary.rename(columns={'<lambda>': 'null_count'}, inplace=True)
print("\nCategorical Feature Summary:")
display(combo_summary)

print("\nFull column list of train:")
print(train_df.columns.tolist())


Numeric Feature Summary:

,nunique,min,max,null_count
road_capacity,6.0,0.0,10.0,0.0



Categorical Feature Summary:


,nunique,null_count
road_combo,4,0



Full column list of train:
['day', 'demand', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'hour', 'minute', 'road_type_ord', 'weather_severity', 'latitude', 'longitude', 'geohash_target_enc', 'geohash_prefix_enc', 'hour_sin', 'hour_cos', 'is_peak_hour', 'is_business_hour', 'is_night', 'hour_squared', 'road_capacity', 'road_combo']


In [5]:
train_df.to_csv(os.path.join(PROC_DIR, 'train_step06.csv'), index=False)
test_df.to_csv(os.path.join(PROC_DIR, 'test_step06.csv'), index=False)

print(f"Saved Train shape: {train_df.shape}")
print(f"Saved Test shape: {test_df.shape}")
print(f"Saved to: {os.path.join(PROC_DIR, 'train_step06.csv')}")
print(f"Saved to: {os.path.join(PROC_DIR, 'test_step06.csv')}")


Saved Train shape: (77299, 22)
Saved Test shape: (41778, 21)
Saved to: C:\traffic-demand-final\pipelining\processed\train_step06.csv
Saved to: C:\traffic-demand-final\pipelining\processed\test_step06.csv


### Step 06 Complete — What Was Done

*   **road_capacity:** A composite numeric feature created by multiplying `road_type_ord` and `NumberofLanes`.
*   **road_combo:** A categorical string feature concatenating `LargeVehicles` and `Landmarks` into four distinct environments: 1_1, 1_0, 0_1, and 0_0.
*   **EDA findings:** Bivariate analysis showed road type and lane count both scale monotonically with demand, justifying their multiplication into a single capacity metric. Multivariate analysis showed a synergistic hierarchy where places allowing heavy vehicles and having landmarks experienced the absolute highest traffic volumes.
*   **Files saved:** Processed datasets saved as `train_step06.csv` and `test_step06.csv` in the `PROC_DIR`.
*   **Next step:** The next phase will involve adding rolling statistics, spatial features, or transitioning toward the final model preparation.
